In [46]:
import requests
import sqlite3
from datetime import datetime
from bs4 import BeautifulSoup
from config import store_id, sku

In [47]:
# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('microcenter_scraper.db')
cursor = conn.cursor()

In [48]:
# Create a table for storing product data
cursor.execute('''
CREATE TABLE IF NOT EXISTS product_data (
    sku TEXT PRIMARY KEY,
    price REAL,
    last_checked DATETIME,
    in_stock INTEGER
)
''')
conn.commit()

In [49]:
# Function to insert or update product data
def upsert_product_data(sku, price, in_stock):
    current_time = datetime.now()
    
    # Check if the SKU already exists
    cursor.execute('SELECT * FROM product_data WHERE sku = ?', (sku,))
    result = cursor.fetchone()

    if result:
        # Update the existing record
        cursor.execute('''
            UPDATE product_data
            SET price = ?, last_checked = ?, in_stock = ?
            WHERE sku = ?
        ''', (price, current_time, in_stock, sku))
    else:
        # Insert a new record
        cursor.execute('''
            INSERT INTO product_data (sku, price, last_checked, in_stock)
            VALUES (?, ?, ?, ?)
        ''', (sku, price, current_time, in_stock))
    
    conn.commit()

In [50]:
search_url = f'https://www.microcenter.com/search/search_results.aspx?Ntt={sku}&Ntx=mode+MatchPartial&Ntk=all&sortby=match&N=0&storeID={store_id}&Change+Store=Change+Store'

In [51]:
# Send a GET request to fetch the page content
response = requests.get(search_url)
if response.status_code != 200:
    raise Exception(f"Failed to load page {search_url}")

# Parse the page content using BeautifulSoup
soup = BeautifulSoup(response.content, 'html.parser')

In [52]:
# Display the raw HTML if needed to inspect
#print(soup.prettify())

# Find the first product listing on the search results page

product_listing = soup.find('li', class_='product_wrapper')

if product_listing:
    # Extract the product title
    title = product_listing.find('a', class_='productClickItemV2')['data-name']
    print(f"Product Title: {title}")

    # Extract the price from the 'data-price' attribute
    price = product_listing.find('a', class_='productClickItemV2')['data-price']
    print(f"Product Price: ${price}")

    # Extract the main product URL
    product_url = f"https://www.microcenter.com{product_listing.find('a', class_='productClickItemV2')['href']}?storeid={store_id}"
    print(f"Product URL: {product_url}")

    # Extract the open box price from the 'clearance' div
    clearance_section = product_listing.find('div', class_='clearance')
    if clearance_section and clearance_section.text.strip():
        open_box_price = clearance_section.find('strong').text.strip()
        print(f"Open Box Available from: {open_box_price}")
    else:
        print("No Open Box items available.")
else:
    print("No product found with the given SKU.")

In [53]:
# Web scraping function to get product details
def scrape_product_data(sku):
    search_url = f'https://www.microcenter.com/search/search_results.aspx?Ntt={sku}&Ntx=mode+MatchPartial&Ntk=all&sortby=match&N=0&storeID={store_id}&Change+Store=Change+Store'

    # Send a GET request to fetch the search results page
    response = requests.get(search_url)
    if response.status_code != 200:
        raise Exception(f"Failed to load page {search_url}")

    # Parse the page content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the first product listing by the class 'product_wrapper'
    product_listing = soup.find('li', class_='product_wrapper')

    if product_listing:
        # Extract the product title
        title = product_listing.find('a', class_='productClickItemV2')['data-name']
        print(f"Product Title: {title}")

        # Extract the price from the 'data-price' attribute
        price = product_listing.find('a', class_='productClickItemV2')['data-price']
        print(f"Product Price: ${price}")

        # Extract the main product URL
        product_url = f"https://www.microcenter.com{product_listing.find('a', class_='productClickItemV2')['href']}?storeid={store_id}"
        print(f"Product URL: {product_url}")

        # Extract the open box price from the 'clearance' div
        clearance_section = product_listing.find('div', class_='clearance')
        if clearance_section and clearance_section.text.strip():
            open_box_price = clearance_section.find('strong').text.strip()
            in_stock = 1
            print(f"Open Box Available from: {open_box_price}")
        else:
            in_stock = 0
            print("No Open Box items available.")
        
        # Store or update product data in the database
        upsert_product_data(sku, price, in_stock)

    else:
        print("No product found with the given SKU.")

In [54]:
# Scrape and store data for a specific SKU
scrape_product_data(sku)

Product Title: 990 PRO 2TB Samsung V NAND 3-bit MLC PCIe Gen 4 x4 NVMe M.2 Internal SSD
Product Price: $179.99
Product URL: https://www.microcenter.com/product/660429/samsung-990-pro-2tb-samsung-v-nand-3-bit-mlc-pcie-gen-4-x4-nvme-m2-internal-ssd?storeid=085
Open Box Available from: $143.96


In [55]:
# Close the connection when done
conn.close()